> ## SOLUTION / ANSWER KEY &mdash; Lab 3.8
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-08-self-attention-sequence.ipynb`](../lab-08-self-attention-sequence.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 3.8 &mdash; Self-Attention Over a Sequence

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Project token embeddings into queries, keys and values
- Run self-attention over a whole sequence at once
- Confirm the attention matrix is row-normalised

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; The transformer block](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-08")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
In a real block, each token embedding **X** is projected by learned matrices into **Q = X.Wq**,
**K = X.Wk**, **V = X.Wv**, then attention runs over the sequence. The **attention matrix** A is
`seq x seq`: row *i* says how much token *i* attends to each token; each row sums to 1. We build it by
hand here so the shapes are unambiguous &mdash; Lab 3.9 extracts exactly this matrix from a real model.

## Build it
Build the projections and the attention matrix.

In [ ]:
import numpy as np
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    d = Q.shape[-1]
    A = softmax(Q @ K.T / np.sqrt(d), axis=-1)
    return A @ V, A

## Run it for real
Run one head of self-attention over a 3-token sequence.

In [ ]:
try:
    import numpy as np
    rng = np.random.default_rng(0)
    X = rng.normal(size=(3, 4))                        # 3 tokens, 4-d embeddings
    Wq = rng.normal(size=(4, 4)); Wk = rng.normal(size=(4, 4)); Wv = rng.normal(size=(4, 4))
    out, A = self_attention(X, Wq, Wk, Wv)
    print("output shape:", out.shape, "| attention matrix shape:", A.shape)
    print("attention matrix (each row sums to 1):\n", np.round(A, 3))
    print("row sums:", np.round(A.sum(axis=1), 3))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__, "--", e)

## What to notice
- The **output** is `(seq_len, value_dim)` &mdash; one new vector per token, mixed from the whole sequence.
- The **attention matrix** is `seq x seq` and every **row sums to 1** &mdash; a per-token distribution over what it looked at.
- The learned `Wq/Wk/Wv` are what training actually optimises; the attention operation itself has **no parameters**.

## Your turn (open task &mdash; no grader)
Give two tokens **identical** embeddings and watch their rows in `A`. Then add more heads:
run `self_attention` a few times with different random `Wq/Wk/Wv` and average the outputs &mdash; that
is multi-head attention in miniature. A "good" answer: you can state the shape of every intermediate
(`Q`, `K`, `A`, output) without running the cell.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
import numpy as np
rng = np.random.default_rng(1)
X = rng.normal(size=(3, 4)); X[1] = X[0]          # tokens 0 and 1 have IDENTICAL embeddings
Wq = rng.normal(size=(4, 4)); Wk = rng.normal(size=(4, 4)); Wv = rng.normal(size=(4, 4))
_, A = self_attention(X, Wq, Wk, Wv)
print("rows 0 and 1 of A are identical:\n", np.round(A[:2], 3))
# multi-head in miniature: average a few heads with different random projections
outs = [self_attention(X, rng.normal(size=(4, 4)), rng.normal(size=(4, 4)), rng.normal(size=(4, 4)))[0]
        for _ in range(4)]
print("averaged 4-head output shape:", np.mean(outs, axis=0).shape)

---
### Key takeaway
You just ran one head of self-attention over a sequence. Stack a few heads and a feed-forward layer and you have a transformer block.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>